# Phase 3: Direct Preference Optimization (DPO) - Quality Alignment

This notebook aligns our model using DPO to prioritize detailed, pedagogically sound, and accurate academic answers over vague, brief, or format-deficient responses.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes xformers
!pip install datasets wandb

In [ ]:
import wandb
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
except Exception:
    pass

try:
    wandb_key = userdata.get('WANDB_API_KEY')
    wandb.login(key=wandb_key)
except Exception:
    pass

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Reduced sequence length for memory efficiency during DPO
dtype = torch.float16
load_in_4bit = True

model_path = "/content/drive/MyDrive/B.Tech-AI-Tutor-7B/models/phase2_merged"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    lora_alpha = 64,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json", 
    data_files="/content/drive/MyDrive/B.Tech-AI-Tutor-7B/data/phase3_dpo.jsonl", 
    split="train"
)
print(f"Loaded DPO dataset with {len(dataset)} preference pairs.")

In [ ]:
from trl import DPOTrainer, DPOConfig

dpo_config = DPOConfig(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 8, # Effective batch size = 16
    warmup_ratio = 0.1,
    num_train_epochs = 1,            # 1 epoch is usually sufficient for preference alignment
    learning_rate = 5e-6,            # Very low learning rate to avoid destroying weights
    beta = 0.1,                      # DPO loss scaling factor
    fp16 = True,
    bf16 = False,
    logging_steps = 5,
    optim = "adamw_8bit",
    lr_scheduler_type = "cosine",
    seed = 42,
    output_dir = "/content/drive/MyDrive/B.Tech-AI-Tutor-7B/checkpoints/phase3_dpo",
    max_length = max_seq_length,
    max_prompt_length = 1024,
    report_to = "wandb" if wandb.run is not None else "none",
)

trainer = DPOTrainer(
    model = model,
    ref_model = None, # Unsloth handles reference model calculations without memory overhead
    args = dpo_config,
    beta = 0.1,
    train_dataset = dataset,
    tokenizer = tokenizer,
)

In [ ]:
trainer.train()

In [ ]:
adapter_path = "/content/drive/MyDrive/B.Tech-AI-Tutor-7B/adapters/phase3_adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Saved final DPO adapter to {adapter_path}")

In [ ]:
merged_path = "/content/drive/MyDrive/B.Tech-AI-Tutor-7B/models/B_Tech_AI_Tutor_7B_7b_final"
model.save_pretrained_merged(merged_path, tokenizer, save_method = "merged_16bit")
print(f"Saved final merged model to {merged_path}")